# Lecture 02 题目讲义

主题：链表、串、KMP、栈、队列

使用方式：先独立思考，再看“关键点”，最后参考代码或解答框架。题目按难度递进。

## Problem 1 · 结构选型

**题目**：有一个长度经常变化的成绩表，需要频繁在已知位置后插入或删除记录，但很少按下标随机访问。应该优先用顺序表还是链表？说明理由。

**关键点**：区分“随机访问”与“局部插删”的主需求。

**解答提要**：优先考虑链表。因为题目强调的是频繁插删，且插删位置一旦找到，改指针即可；顺序表则可能需要整体搬移元素。若需求改为频繁访问第 $k$ 个元素，则顺序表更优。

## Problem 2 · 单链表插入与删除手推

**题目**：设单链表为 `head -> 8 -> 12 -> 20 -> 31 -> None`。

1. 在结点 `12` 后插入新结点 `15`。
2. 删除结点 `20`。

**关键点**：改链顺序不能错；插入先接后继，再接前驱；删除要保住前驱。

**解答提要**：

- 插入：`new.next = cur.next`，再 `cur.next = new`
- 删除：若 `cur.next` 就是待删结点，则 `cur.next = cur.next.next`
- 结果链表：`head -> 8 -> 12 -> 15 -> 31 -> None`

In [ ]:
class Node:
    def __init__(self, data, next=None):
        self.data = data
        self.next = next

def insert_after(cur, value):
    node = Node(value)
    node.next = cur.next
    cur.next = node

def delete_after(cur):
    if cur is None or cur.next is None:
        return None
    removed = cur.next
    cur.next = removed.next
    return removed.data


## Problem 3 · 删除首个值为 x 的结点

**题目**：实现函数 `remove_first(head, x)`，删除单链表中首个值为 `x` 的结点，返回新的头指针。

**关键点**：首结点可能就是要删的目标；遍历时通常需要 `prev` 和 `cur` 两个指针。

**解答提要**：

1. 若 `head` 为空，直接返回。
2. 若 `head.data == x`，返回 `head.next`。
3. 否则遍历，找到后令 `prev.next = cur.next`。

In [ ]:
def remove_first(head, x):
    if head is None:
        return None
    if head.data == x:
        return head.next

    prev, cur = head, head.next
    while cur is not None:
        if cur.data == x:
            prev.next = cur.next
            break
        prev, cur = cur, cur.next
    return head


## Problem 4 · 子串、子序列、字符串操作

**题目**：设 `s = "algorithm"`。

1. 判断 `"gori"` 是否为 `s` 的子串。
2. 判断 `"agtm"` 是否为 `s` 的子序列。
3. 说明 Python 中为什么不建议在大循环里反复做 `ans += piece`。

**关键点**：子串要求连续；子序列只要求相对顺序不变；`str` 在 Python 中不可变。

**解答提要**：`"gori"` 不是子串，`"agtm"` 是子序列。大量拼接会不断产生新字符串，常用 `list` 收集后 `"".join(parts)`。

## Problem 5 · 朴素模式匹配

**题目**：主串 `T = "ababcabcacbab"`，模式串 `P = "abcac"`。请手推朴素匹配的过程，并指出哪些比较发生了重复。

**关键点**：失配后模式串整体右移一位，因此主串中的某些字符会重复参与比较。

**解答提要**：从起点 `0` 对齐时，前两个字符成功；失配后重新从起点 `1`、`2` 对齐，会再次比较前面已经见过的 `a`、`b`。这正是朴素算法最坏达到 `O(nm)` 的原因。

## Problem 6 · 计算 KMP 的 lps 数组

**题目**：求模式串 `P = "ababaca"` 的 `lps`（或前缀函数长度数组）。

**关键点**：对每个前缀，找“最长相等前后缀”的长度。

**答案**：`[0, 0, 1, 2, 3, 0, 1]`

**说明**：在最后的 `a` 之前读到 `c` 时，会发生连续回退：从长度 `3` 回退到 `1`，再回退到 `0`。

In [ ]:
def build_lps(p):
    lps = [0] * len(p)
    j = 0
    for i in range(1, len(p)):
        while j > 0 and p[i] != p[j]:
            j = lps[j - 1]
        if p[i] == p[j]:
            j += 1
        lps[i] = j
    return lps

print(build_lps('ababaca'))


## Problem 7 · KMP 查找首个匹配位置

**题目**：实现 `kmp_find(text, pattern)`，返回 `pattern` 在 `text` 中第一次出现的位置；若不存在，返回 `-1`。

**关键点**：主串指针 `i` 不回退；失配时通过 `lps` 调整模式串指针 `j`。

**解答提要**：先构造 `lps`，扫描主串；若失配且 `j > 0`，令 `j = lps[j - 1]`；若匹配则 `j += 1`；当 `j == len(pattern)` 时返回起点。

In [ ]:
def kmp_find(text, pattern):
    if pattern == '':
        return 0
    lps = build_lps(pattern)
    j = 0
    for i, ch in enumerate(text):
        while j > 0 and ch != pattern[j]:
            j = lps[j - 1]
        if ch == pattern[j]:
            j += 1
        if j == len(pattern):
            return i - len(pattern) + 1
    return -1

print(kmp_find('ababcabcacbab', 'abcac'))


## Problem 8 · 栈与队列应用辨析

**题目**：下面场景更适合栈还是队列？简述理由。

1. 括号匹配
2. 浏览器回退
3. 打印任务调度
4. 图的层次遍历

**关键点**：看“最近状态优先”还是“先来先服务”。

**答案**：

- 括号匹配：栈
- 浏览器回退：栈
- 打印任务调度：队列
- 图的层次遍历：队列

## Problem 9 · 合法出栈序列

**题目**：入栈序列固定为 `1, 2, 3, 4, 5`。判断序列 `2, 1, 5, 4, 3` 是否可能是一个合法出栈序列。

**关键点**：直接模拟，不要凭感觉猜。

**解答提要**：依次入栈直到栈顶等于当前需要弹出的元素，能弹则弹。该序列合法：先入 `1,2` 后弹 `2,1`，再入 `3,4,5` 后弹 `5,4,3`。

## Problem 10 · 课后提高

**题目**：设计一个循环队列，给出判空、判满条件，并说明为什么循环队列能避免“假溢出”。

**关键点**：顺序队列在头指针前移后会留下空位；循环队列通过取模复用这些空位。

**解答提要**：若采用“浪费一个单元”的设计，则判空条件为 `front == rear`，判满条件为 `(rear + 1) % maxsize == front`。循环队列让队尾回到数组前端，从而复用之前出队腾出的空间。

## Checklist

做完本讲题目后，应当能够：

- 解释顺序表与链表的适用边界
- 画出单链表插删时的指针变化
- 手推朴素匹配与 KMP 的关键步骤
- 说明栈与队列分别适合哪些问题